# alphaPhos canonical phospho-proteomics analysis pipeline

This notebook walks through the **recommended end-to-end workflow** for a
phospho-MS analysis: from a raw Spectronaut PSM report to a volcano plot of
differentially regulated sites.

It uses the **EGF benchmark dataset** (6-run nanoPhos DIA, 3 withEGF + 3
woEGF) as a worked example. Swap in your own report path + condition map
to use the same pipeline on new data.

## Pipeline at a glance

| Step | Tool | What it does |
|---|---|---|
| 1 | `alphaphos.io.read_spectronaut` | PSM ingestion + dedup + contaminant filter |
| 2 | (your `condition_df`) | sample → condition map |
| 3 | `alphaphos.preprocess.collapse_sites` | precursors → phosphosites, condition-aware Class I |
| 4 | `alphaphos.preprocess.to_anndata` | wide quants + phospho-specific var/uns metadata |
| 5 | `apt.pp.filter_data_completeness` | drop low-coverage sites |
| 6 | `alphaphos.preprocess.impute_hybrid` | per-cell MAR (KNN) / MNAR (Gaussian) split |
| 7 | `apt.tl.pca` + `apt.pl.plot_pca` | sample-level QC |
| 8 | `apt.tl.diff_exp_ebayes` | limma via `inmoose` (pure Python, no R) |
| 9 | `apt.pl.volcano` + top hits | results |
| 10 | `alphaphos.kinase.predict_kinases` | per-site upstream-kinase prediction (Yaffe PWMs) |
| 11 | `alphaphos.kinase.kinase_enrichment_from_diffexp` + `kinase_mea` | KSEA — per-kinase activity z-scores / NES |
| 12 | `adata.write_h5ad` | persist for downstream / sharing |

**Division of labor:** alphaPhos owns the phospho-specific parts (steps 1–4, 6).
alphapepttools owns the generic downstream (5, 7–9). Both operate on the same
`AnnData` object — handoff is zero-friction.


In [ ]:
from pathlib import Path
import sys
import warnings; warnings.filterwarnings("ignore")
import logging; logging.basicConfig(level=logging.INFO, format="%(message)s")

import numpy as np
import pandas as pd

# alphaPhos — phospho-specific upstream
import alphaphos
from alphaphos.io import read_spectronaut
from alphaphos.preprocess import (
    collapse_sites,
    to_anndata,
    impute_hybrid,            # MAR/MNAR per-cell split (p30 default)
    impute_knn_site_based,    # legacy-parity alternative (Pearson r=1.000 vs R-limma)
)

# alphapepttools — generic downstream
import alphapepttools as apt

# --- Preflight: ensure the kernel is on the right env -----------------------
print(f"Python interpreter: {sys.executable}")
print(f"alphaPhos:          {alphaphos.__version__}")
print(f"alphapepttools:     {apt.__version__ if hasattr(apt,'__version__') else '?'}")
# Older alphapepttools (<0.2.1) lacks `keep_strategy` in filter_data_completeness.
# If you hit a TypeError on that arg later, upgrade with:
#   !python -m pip install --upgrade "git+https://github.com/MannLabs/alphapepttools.git"
# then RESTART THE KERNEL.
import inspect as _inspect
_filter_params = _inspect.signature(apt.pp.filter_data_completeness).parameters
assert "keep_strategy" in _filter_params, (
    "alphapepttools is too old (no keep_strategy on filter_data_completeness). "
    "Run !python -m pip install --upgrade "
    "git+https://github.com/MannLabs/alphapepttools.git "
    "and restart the kernel."
)

ROOT = Path("D:/Projects/alphaPhos")
PSM_TSV = ROOT / "test_data/benchmark/EGF_report_new_ms2.tsv"   # change for your data
OUT_DIR = ROOT / "test_data/benchmark_output"
OUT_DIR.mkdir(exist_ok=True)


## §1. Read the Spectronaut PSM report

`read_spectronaut` is the gate. It loads the report, normalizes column names,
and applies four PSM-level filters with sensible phospho-aware defaults:

- `drop_decoys=True` — drop `EG.IsDecoy` rows
- `pg_qvalue_max=0.01` — protein-group FDR cap
- `top_n_attribution=True` — Spectronaut over-export dedup (drops redundant
   peptide rows that map to the same site but encode different candidate
   positions). Validated against SN classI at Pearson r=0.99.
- `drop_contaminants=True` — filter PSMs whose protein group is entirely
   common contaminants (trypsin, BSA, keratins). Uses the bundled MaxQuant
   `contaminants.fasta`. Drops ~0.5% of rows on a typical phospho run.

The function records every stage's row count in `df.attrs` for audit.


In [ ]:
df = read_spectronaut(PSM_TSV)

# read_spectronaut records per-stage row counts in df.attrs for audit.
a = df.attrs
# Newer versions may or may not populate every key -- fall back gracefully.
for label, key in [
    ("Loaded rows",                     "n_rows_loaded"),
    ("  after top-N attribution",       "n_rows_after_top_n"),
    ("  after contaminant filter",      "n_rows_after_contaminant_filter"),
    ("Final rows returned",             "n_rows_returned"),
]:
    if key in a:
        print(f"{label:>36s}:  {a[key]:>9,}")
if "alphaphos_quant_column" in a:
    print(f"{'Quant column used':>36s}:  {a['alphaphos_quant_column']}")
df.head(3)


## §2. Sample → condition mapping

`condition_df` is a two-column DataFrame mapping each run (`sample`) to its
biological condition. Extra columns (batch, donor, etc.) are propagated
into `adata.obs` and can be used downstream for filtering / coloring /
covariates.


In [ ]:
SAMPLES = sorted(df["R.FileName"].unique())
print(f"{len(SAMPLES)} samples in the report:")
for s in SAMPLES:
    print(f"  {s}")

# Build condition map by parsing the sample name
condition_df = pd.DataFrame({
    "sample": SAMPLES,
    "condition": ["withEGF" if "withEGF" in s else "woEGF" for s in SAMPLES],
})
print()
print(condition_df.to_string(index=False))


## §3. Collapse precursors to phosphosites

`collapse_sites` runs the full Hogrebe-style collapse with the
SN-validated defaults:

- `aggregation_method='sum'` — best match to Spectronaut's native PTM
   site report (Pearson r=0.99 on per-cell log2 quant; `'median'` introduces
   an intensity-dependent bias).
- `localization_strategy='condition'` — site collapse runs `global_max`
   upstream, then the per-condition Class-I majority mask is applied
   (`classI_cutoff=0.75`, `condition_threshold=0.50`). Site-set Jaccard 0.98
   vs SN classI on the EGF dataset.
- `return_decision_table=True` — optional Class-I per-condition decision
   table for downstream auditing.

If you have a proteome FASTA, pass `fasta_path="proteome.fasta"` and kinase
windows (`±7` residues around each phospho-S/T/Y) will be auto-computed —
needed for the kinase-Library / KSEA workflows in `alphaphos.kinase`.


In [ ]:
# collapse_sites now returns an AnnData directly (the earlier 3-tuple return +
# separate to_anndata step has been consolidated).
adata = collapse_sites(df, condition_df=condition_df)

print(f"AnnData shape: {adata.shape} (samples x sites)")
print(f"Layers: {list(adata.layers.keys())}")
print(f"obs cols: {list(adata.obs.columns)}")
print(f"var cols ({len(adata.var.columns)}): {list(adata.var.columns)}")
print()
if "alphaphos" in adata.uns:
    ns = adata.uns["alphaphos"]
    for k in ("version", "processing_timestamp"):
        if k in ns:
            print(f"  uns['alphaphos']['{k}']  = {ns[k]}")
    if "stats" in ns:
        print("\nCollapse stats:")
        for k, v in ns["stats"].items():
            print(f"  {k}: {v}")


## §4. AnnData assembly (folded into §3)

As of the current alphaPhos API, `collapse_sites` produces a fully-assembled
`AnnData` in one call — the old `sites -> to_anndata` two-step is gone. The
per-site metadata (structural + motif flags + site QC), per-sample
metadata, and `uns['alphaphos']` provenance now populate directly on the
AnnData returned by `collapse_sites`.


## §5. Filter low-coverage sites

`apt.pp.filter_data_completeness` drops sites that don't meet a
completeness threshold. Two flags drive its behaviour:

- `max_missing=0.3` — at most 30% of samples in a group can be missing
- `keep_strategy='any'` (lenient) — keep if ≥1 group passes;
  `'all'` (strict) — require every group to pass

The lenient default matches Dublin's `filter_phosphosites(how='condition',
cutoff=0.7)` (presence ≥0.7 in at least one condition).


In [ ]:
adata = apt.pp.filter_data_completeness(
    adata,
    max_missing=0.3,
    group_column="condition",
    keep_strategy="any",
    action="drop",
)

# missingness stats post-filter
n_total = adata.X.size
n_miss = int(np.isnan(adata.X).sum())
print(f"adata after filter: {adata.shape}")
print(f"Missing cells:      {n_miss:,} / {n_total:,} ({n_miss/n_total*100:.1f}%)")


## §6. Hybrid imputation (MAR + MNAR)

The key phospho-specific step that alphapepttools doesn't ship.

`impute_hybrid` splits per-cell:

- **MNAR (Missing Not At Random)** — site mean intensity below the 30th
  percentile of the dataset. The value is below the limit of detection.
  Imputed via per-sample downshifted Gaussian (`μ − 1.8σ`, width `0.3σ`)
  — the Perseus convention.
- **MAR (Missing At Random)** — everywhere else. The site IS detectable;
  the missing value is a technical miss. Imputed via **site-based KNN**
  (transposed direction; `n_neighbors=int(sqrt(n_samples))`,
  `weights='uniform'`).

Why hybrid and not pure KNN or pure Gaussian:
- **Pure KNN** correctly imputes MAR but *inflates* MNAR (borrows from
  detectable neighbours, dampening true low-abundance fold-changes).
- **Pure Gaussian** correctly imputes MNAR but *inflates fold-changes
  for high-abundance sites* (where the missing cell isn't really below
  LOD). On the EGF benchmark, pure Gaussian gives EGFR Y1172 a logFC of
  −7.55, vs the correct −6.59 from KNN.
- **Hybrid** uses each where it applies. EGFR Y1172 stays at −6.59
  (high abundance → MAR → KNN), and ~30% of low-abundance MNAR sites
  get correctly downshifted instead of inflated.

`return_audit=True` returns a per-cell DataFrame showing which strategy
was used — useful for debugging or for stratification plots.


In [ ]:
adata, audit = impute_hybrid(
    adata,
    mnar_threshold_percentile=30.0,    # default; bottom 30% of sites get Gaussian
    return_audit=True,
)

# impute_hybrid writes to layers['intensity_log2'] only.  apt.tl.pca (and
# other downstream tools) reads from adata.X by default, so copy the imputed
# layer over.
adata.X = adata.layers["intensity_log2"].copy()

n_mar = (audit["strategy"] == "MAR_KNN").sum()
n_mnar = (audit["strategy"] == "MNAR_Gaussian").sum()
print(f"Imputed {len(audit):,} cells total:")
print(f"  MAR (site-KNN):     {n_mar:>7,}  ({n_mar / len(audit) * 100:.1f}%)")
print(f"  MNAR (Gaussian):    {n_mnar:>7,}  ({n_mnar / len(audit) * 100:.1f}%)")
print()
print(f"Remaining NaNs after imputation: {int(np.isnan(adata.X).sum())}  (should be 0)")


## §7. PCA — sample-level QC

`apt.tl.pca` adds the embedding to `adata.obsm["X_pca"]` and the explained
variance to `adata.uns["pca"]`. `apt.pl.plot_pca` produces a PC1/PC2 scatter
colored by any obs column.

Sanity checks at this step:
- Do replicates of each condition cluster together?
- Is PC1 capturing the condition contrast (vs e.g. a batch effect)?
- Any obvious outliers?


In [ ]:
import matplotlib.pyplot as plt

apt.tl.pca(adata, n_comps=2)

# alphapepttools namespaces results by method + dim_space. The defaults
# (method='pca', dim_space='obs') write to:
#   adata.obsm['X_pca_obs']
#   adata.varm['PCs_pca_obs']
#   adata.uns['variance_pca_obs']  -> {'variance_ratio': [...], 'variance': [...]}
var = adata.uns["variance_pca_obs"]["variance_ratio"]
print(f"PC1 / PC2 explained variance: {var[0]*100:.1f}% / {var[1]*100:.1f}%")

fig, ax = plt.subplots(figsize=(6, 5))
apt.pl.plot_pca(adata, color_map_column="condition", ax=ax)
plt.show()


## §7b. Advanced PCA with `alphaphos.dimred` — loadings analysis + imputation-impact QC

The `ap.dimred` module ships three PCA backends and a set of accessors for
downstream analysis:

- **`pca(adata, handle_missing=...)`** — three methods:
  - `"error"` (default) — standard SVD via sklearn; requires a complete matrix
  - `"nipals"` — Wold 1966 iterative PCA; **skips NaN natively** (metabolomics standard)
  - `"ppca"` — Tipping–Bishop 1999 probabilistic PCA with EM; principled MV handling
- **`compare_imputation_impact(adata_raw, adata_imputed)`** — flags whether
  the imputation step distorts the sample-space structure vs a
  missing-value-aware PCA on the raw matrix. Sanity check before you
  commit to imputation.
- **`get_pca_dataframe(adata)`** — plot-ready DataFrame (PC1..PCK + `.obs`).
- **`loadings_for_enrichment(adata)`** — canonicalized loadings ready for
  `ap.enrichment.gsea` / `.kinase_activity` / `.ora` with **no wrappers**
  (see §12b below).
- **`feature_variance_contribution(adata)`** — variance-weighted per-site
  importance across the top-K PCs.

Results are attached to `adata.obsm["X_pca"]`, `.varm["PCs"]`, and
`.uns["pca"]` (scanpy convention).


In [ ]:
import alphaphos as ap

# 1. Standard PCA on the imputed matrix. Runs sklearn SVD under the hood
#    (method="standard") since imputation removed all NaN.
adata_pca = ap.dimred.pca(adata, n_components=5, layer="intensity_log2", copy=True)
vr = adata_pca.uns["pca"]["variance_ratio"]
print(f"[dimred.pca standard]   PC1..PC5 var ratio: "
      f"{[f'{v:.3f}' for v in vr]}  method={adata_pca.uns['pca']['method']!r}")

# 2. Plot-ready DataFrame -- one row per sample, PC1..PCK + all .obs columns.
pc_df = ap.dimred.get_pca_dataframe(adata_pca, n_components=3)
print("\nPC coordinates (first 3 PCs):")
print(pc_df[["PC1", "PC2", "PC3", "condition"]].to_string())

# 3. Imputation-impact QC: rebuild a raw (with-NaN) adata via the same
#    collapse + filter, then run NIPALS on it and compare to the imputed PCA.
#    If the top-K PCs correlate near 1.0, imputation preserved the sample-
#    space structure.  If not, imputation is distorting the biology.
adata_raw = collapse_sites(df, condition_df=condition_df)
adata_raw = apt.pp.filter_data_completeness(
    adata_raw, max_missing=0.3, group_column="condition",
    keep_strategy="any", action="drop",
)
if "intensity_log2" not in adata_raw.layers:
    adata_raw.layers["intensity_log2"] = adata_raw.X.copy()

report = ap.dimred.compare_imputation_impact(
    adata_raw, adata, n_components=5, method_raw="nipals",
)
print(f"\n[compare_imputation_impact] {report['verdict']}")
print(f"  per-PC |r| (imputed vs raw NIPALS): "
      f"{[f'{v:.3f}' for v in report['per_pc_correlation']]}")

# 4. Loading analysis: which sites drive PC1?
loadings_wide = ap.dimred.get_pca_loadings(adata_pca, n_components=3, top_n_per_pc=10)
print("\nTop-10 |loading| sites per PC (wide view):")
print(loadings_wide.round(3).to_string())

# 5. Variance-weighted per-site importance across the top-5 PCs.
contrib = ap.dimred.feature_variance_contribution(adata_pca, n_components=5)
print("\nTop-10 sites by variance contribution across PC1..PC5:")
print(contrib.head(10).round(4).to_string())


## §8. Differential analysis — limma via inmoose

`apt.tl.diff_exp_ebayes` wraps `inmoose` (pure-Python Bioconductor limma port).
Same algorithm as R limma; same numerical results to floating-point precision.
No R subprocess; works on any platform.

Pass:
- `between_column` — the obs column whose values define groups
- `comparison=(treatment, control)` — `logFC = mean(treatment) - mean(control)`,
   so positive = up in treatment

Result is a long DataFrame with columns including:
- `protein` — the site identifier (alphaPhos `PTM_Collapse_key`)
- `log2fc` — moderated log2 fold-change
- `p_value`, `fdr` — moderated-t p-value and BH-adjusted
- `stat`, `B` — moderated-t statistic, log-odds of differential expression


In [ ]:
treatment, control = "woEGF", "withEGF"  # logFC = mean(treatment) - mean(control)

comp_id, results = apt.tl.diff_exp_ebayes(
    adata,
    between_column="condition",
    comparison=(treatment, control),
)

# Add significance flag (adj.P < 0.05 & |logFC| > 0.585 == ~1.5x fold-change)
results["sig"] = (results["fdr"] < 0.05) & (results["log2fc"].abs() > 0.585)

print(f"Comparison ID: {comp_id}")
print(f"Features tested: {len(results):,}")
print(f"Significant hits (FDR<0.05, |logFC|>0.585): {int(results['sig'].sum()):,}")
print(f"  up in {treatment}:   "
      f"{int(((results['sig']) & (results['log2fc'] > 0)).sum()):,}")
print(f"  up in {control}: "
      f"{int(((results['sig']) & (results['log2fc'] < 0)).sum()):,}")


## §9. Volcano plot + top hits

Standard volcano: x = logFC, y = −log10(FDR), colored by significance.

The top hits should include the textbook EGF signaling readouts:
- **EGFR autophosphorylation sites**: Y1172, Y1197 — strong negative logFC
  in woEGF (= up under EGF) is the canonical positive control.
- **Downstream signaling**: SHC1, GAB1, CRK, PIK3 components, MAPK pathway.
- **Negative-feedback regulators**: TSC2, INPPL1, etc.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 6))
apt.pl.volcano(
    results,
    x_column="log2fc",                    # results has 'log2fc' column already
    y_column="-log10(fdr)",               # FDR-axis volcano (standard for proteomics)
    x_thresholds=(-0.585, 0.585),         # |logFC| > 0.585 (~1.5x)
    y_thresholds=-np.log10(0.05),         # adj.P < 0.05
    ax=ax,
)
plt.tight_layout()
plt.show()


In [ ]:
# Top 15 hits by FDR — annotate with gene/site for biology check
top = results.nsmallest(15, "fdr")[["protein", "log2fc", "p_value", "fdr"]].copy()
top["log2fc"] = top["log2fc"].round(3)
top["p_value"] = top["p_value"].apply(lambda x: f"{x:.2e}")
top["fdr"] = top["fdr"].apply(lambda x: f"{x:.2e}")
top.reset_index(drop=True, inplace=True)
top


## §10. Per-site kinase prediction (Yaffe Kinase Library)

**Per-site PWM scoring — answers: "for *this specific site*, which
kinase is the most likely regulator?"** This is sequence-based prediction
of upstream kinase(s) using the Yaffe Kinase Library PWMs (Johnson et al.
*Nature* 2023) — 311 Ser/Thr + 78 tyrosine kinases. Per-site PWM
prediction; NOT kinase activity / enrichment (that's §11).

**Prerequisite:** `kinase_sequence` must be populated in `adata.var`. That
happens automatically when you pass `fasta_path="proteome.fasta"` to
`collapse_sites()`. If your `adata` doesn't have this column, re-run §3
with a FASTA path.

Two functions:
- `score_kinases(adata)` writes the full (sites × kinases) score matrices
  to `adata.varm['kinase_score_ser_thr']` and `adata.varm['kinase_score_tyrosine']`.
- `predict_kinases(adata, top_k=5)` returns a per-site DataFrame with the
  top-k kinases. Calls `score_kinases` automatically on first run.

**What this answers:** "For *this specific site*, which kinase is most
likely the regulator?" — complementary to the network-based KSEA
question ("which kinase *activities* changed?") which is the planned
`alphaphos.ksea` module.


In [ ]:
from alphaphos.kinase import predict_kinases

# Only run if we have kinase_sequence (i.e. collapse was run with fasta_path)
if "kinase_sequence" not in adata.var.columns:
    print("Skipping — no kinase_sequence column in adata.var.")
    print("Re-run §3 with collapse_sites(..., fasta_path='proteome.fasta')")
else:
    pred = predict_kinases(adata, top_k=5)

    # Summary
    info = adata.uns["alphaphos_kinase"]
    print(f"Scored {info['n_sites_scored']:,} sites "
          f"({info['n_ser_thr_scored']:,} S/T + {info['n_tyrosine_scored']:,} Y); "
          f"{info['n_sites_dropped']:,} dropped (invalid sequences)")
    print()

    # Attach the top1 prediction to adata.var for easy filtering/coloring
    adata.var["top1_kinase"] = pred["top1_kinase"]
    adata.var["top1_kinase_score"] = pred["top1_score"]
    adata.var["kin_type"] = pred["kin_type"]

    # Show the top-15 differentially regulated sites, annotated with their
    # predicted upstream kinases
    top_hits = (results.nsmallest(15, "fdr")
                       .set_index("protein")
                       .join(pred[["top1_kinase", "top1_score", "top_kinases"]],
                             how="left")
                       [["log2fc", "fdr", "top1_kinase", "top1_score", "top_kinases"]])
    top_hits.round(3)


## §11. Kinase enrichment / KSEA (Yaffe Kinase Library)

Where §10 asked "for *this site*, which kinase?", §11 asks **"across the
whole experiment, which kinase *activities* are changed?"** This is
classical KSEA — the headline kinase-level statistic in most phospho
papers.

`alphaphos.kinase.enrichment` exposes three complementary frameworks:

- **`kinase_enrichment_from_diffexp`** — Fisher's exact, per direction
  (up- and down-regulated sites separately). Takes the diff_exp result
  as input. Returns: per-kinase log2 frequency factor + Fisher p-value,
  per direction. **Most direct equivalent of classical KSEA.**

- **`kinase_mea`** — GSEA-style (weighted Kolmogorov-Smirnov via
  `gseapy`). Operates on the full ranked list (no hard threshold).
  Returns: per-kinase NES + p-value + FDR. **Most rigorous** (uses every
  site's rank, not a binarised threshold).

- **`kinase_enrichment_binary`** — Fisher's exact, custom foreground vs
  background (e.g. for a PCA cluster, or a pathway-curated site set).

For the EGF dataset we'd expect:
- **RTK / Src-family** activity *up* (EGFR autophos drives the cascade)
- **AKT / mTOR / RSK** activity *up* (canonical EGF downstream)
- **MAPK family** (ERK1/2) activity *up* (EGF→RAS→RAF→MEK→ERK)
- Phosphatases or feedback regulators *down*


In [ ]:
from alphaphos.kinase import kinase_enrichment_from_diffexp, kinase_mea

if "kinase_sequence" not in adata.var.columns:
    print("Skipping — no kinase_sequence column (re-run §3 with fasta_path).")
else:
    # 1. Classical KSEA (Fisher per direction) — runs in seconds
    print("Running Fisher-based KSEA (kinase_enrichment_from_diffexp)...")
    ksea = kinase_enrichment_from_diffexp(
        diff_results=results,
        sequence_lookup=adata.var["kinase_sequence"],
        id_col="protein",
        lfc_col="log2fc",
        pval_col="fdr",
        lfc_thresh=0.585,
        pval_thresh=0.05,
        kl_method="percentile",
        kl_thresh=90,
    )

    # Show the top activated/inhibited kinases in each kin_type
    for kt, df_kt in ksea.items():
        print(f"\n=== {kt} — top 10 by 'most significant' direction ===")
        top = (df_kt
                 .sort_values("most_sig_fisher_adj_pval")
                 .head(10)
                 [["most_sig_direction", "most_sig_log2_freq_factor",
                   "most_sig_fisher_adj_pval", "fg_counts_upreg", "fg_counts_downreg"]])
        print(top.round(3))

    # Stash in adata.uns for downstream / sharing
    adata.uns["alphaphos_ksea_fisher"] = {k: v.copy() for k, v in ksea.items()}


In [ ]:
if "kinase_sequence" not in adata.var.columns:
    print("Skipping GSEA-style MEA -- no kinase_sequence column (re-run §3 with fasta_path).")
else:
    # 2. GSEA-style MEA -- uses the full rank, no threshold. Slower (permutation).
    print("Running GSEA-style MEA (kinase_mea, 1000 permutations)...")
    mea = kinase_mea(
        diff_results=results,
        sequence_lookup=adata.var["kinase_sequence"],
        id_col="protein",
        rank_col="log2fc",
        kl_method="percentile",
        kl_thresh=90,
        permutation_num=1000,
        seed=42,
    )

    # Top activated (positive NES) and inhibited (negative NES) kinases per type
    for kt, df_kt in mea.items():
        nonna = df_kt.dropna(subset=["NES"]).sort_values("NES", ascending=False)
        print(f"\n=== {kt} -- MEA NES leaders ===")
        print("Top 10 ACTIVATED (positive NES):")
        print(nonna.head(10)[["NES", "p-value", "FDR", "Subs fraction"]].round(3))
        print("\nTop 10 INHIBITED (negative NES):")
        print(nonna.tail(10)[::-1][["NES", "p-value", "FDR", "Subs fraction"]].round(3))

    adata.uns["alphaphos_ksea_mea"] = {k: v.copy() for k, v in mea.items()}


## §12. Network-based KSEA (Saez-Rodriguez / decoupler-py + OmniPath)

Where §11 used **sequence-based PWMs** (Yaffe Kinase Library) to identify
kinase activity, §12 uses **curated kinase-substrate databases**
(PhosphoSitePlus, SIGNOR, KEA, NetworKIN, ...) via OmniPath as the
substrate set priors, and decoupler-py's statistical methods to score
per-kinase activity.

The two approaches are complementary:

| Question | Method |
|---|---|
| "What does the *motif* around this site predict?" | PWM-based (§11) |
| "What does the *biological literature* say this site is regulated by?" | Network-based (§12) |

Both should converge on the same conclusions for well-studied pathways.
For novel sites with no DB record, §11 still works but §12 can't score
them (no substrate set to match against).

`alphaphos.ksea` exposes four methods:

- **`kinase_activity_ulm`** — Univariate Linear Model (the classical
  KSEA z-score equivalent; recommended default)
- **`kinase_activity_mlm`** — Multivariate (joint fit; controls for
  shared substrates)
- **`kinase_activity_ora`** — Over-Representation Analysis (Fisher tail)
- **`kinase_activity_gsea`** — GSEA on the full rank


In [ ]:
from pathlib import Path
# alphaphos.ksea was moved into alphaphos.enrichment.ksea; both entrypoints
# remain re-exported at alphaphos.enrichment.
from alphaphos.enrichment import fetch_omnipath_ks_network, kinase_activity

# 1. Fetch (and cache) the curated kinase-substrate network for human.
#    First call hits OmniPath over the network (10-30s); subsequent calls
#    read the parquet cache.
cache = OUT_DIR / "ks_human.parquet"
try:
    net = fetch_omnipath_ks_network(
        organism="human",
        cache_path=cache,
    )
    print(f"KS network: {len(net):,} edges, {net['source'].nunique():,} kinases, "
          f"{net['target'].nunique():,} unique sites")

    # 2. Run ULM via kinase_activity (the top-level entrypoint).
    print("\nRunning network-based KSEA (ULM)...")
    ksea_net = kinase_activity(
        results,
        stat_col="log2fc",
        network=net,
        method="ulm",
        min_substrates=5,
        organism="human",
        key_column="protein",
    )
    print(f"KSEA scored {len(ksea_net):,} kinases with >= 5 substrates.")
    top_act = ksea_net.sort_values("score", ascending=False).head(15)
    top_inh = ksea_net.sort_values("score").head(15)
    print("\nTop 15 ACTIVATED kinases (score desc):")
    print(top_act.round(3).to_string())
    print("\nTop 15 INHIBITED kinases (score asc):")
    print(top_inh.round(3).to_string())

    adata.uns["alphaphos_ksea_network_ulm"] = ksea_net.copy()
except Exception as e:
    print(f"Skipping network KSEA: {type(e).__name__}: {e}")
    print("  (needs `pip install alphaPhos[enrichment]` + network access to OmniPath)")


## §12b. Site-set enrichment against the PTM functional database (`alphaphos.enrichment`)

Where §11/§12 answered **"which kinase activities changed?"**, this section
answers **"is my hit set enriched for functionally important phosphosites?"**
The PTM functional DB integrates 8 curated sources into ~504k phospho-relations
covering: **functional-effect** annotations (activating / inhibitory), **PPI
disruption / induction**, **disease-variant** overlap (ClinVar / TCGA), and
**functional score** bins (Ochoa 2020).

Three complementary tests exposed under `alphaphos.enrichment`:

- **`ora(hits, background, libraries=...)`** — Fisher's exact per set.
  Categorical: hit vs non-hit.
- **`gsea(ranked_stats, libraries=...)`** — Preranked GSEA on the full
  ranked list (Subramanian 2005; permutation FDR).
- **`emit_libraries(...)` / `load_libraries(...)`** — generate / read the
  site-set GMT libraries emitted from the PTM DB.

The **new bridge from §7b**: `ap.dimred.loadings_for_enrichment(adata)` returns
loadings indexed by canonical `Protein_AApos` site IDs — drops straight into
`gsea` and `ora` with no wrappers. That means you can ask "which PTM
sets drive PC1?" as easily as "which PTM sets drive the EGF response?".


In [ ]:
from alphaphos.enrichment import (
    emit_libraries, load_libraries, ora, gsea,
    parse_alphaphos_key, site_id,
)

# 1. Emit + load site-set libraries from the bundled PTM functional DB.
try:
    LIB_DIR = OUT_DIR / "ptm_libraries"
    LIB_DIR.mkdir(exist_ok=True)
    emit_libraries(LIB_DIR, min_set_size=5)
    libraries = load_libraries(LIB_DIR)
    total_sets = sum(len(sets) for sets in libraries.values())
    print(f"[PTM-DB]  {len(libraries)} libraries, {total_sets:,} sets")
except Exception as e:
    libraries = None
    print(f"Skipping PTM-DB enrichment: {type(e).__name__}: {e}")

# 2. Canonicalize the diff-exp result to Protein_AApos site IDs so it matches
#    the library membership scheme.
def _canonical_id(alphaphos_key: str) -> str | None:
    p = parse_alphaphos_key(alphaphos_key)
    return site_id(p.protein, p.residue, p.position) if p else None

canon = results["protein"].map(_canonical_id)
mask = canon.notna()
results_canon = results.loc[mask].copy()
results_canon["site_id"] = canon.loc[mask].values

# 3. ORA on FDR<0.05, |log2fc|>0.585 hits vs all measured sites as background.
if libraries is not None:
    hits = results_canon.loc[results_canon["sig"], "site_id"].tolist()
    background = results_canon["site_id"].tolist()
    print(f"\n[ORA]  hits={len(hits):,}  background={len(background):,}")
    ora_res = ora(hits, background, libraries=libraries, min_overlap=2)
    if len(ora_res):
        cols = [c for c in ("library", "set_name", "n_overlap", "odds_ratio",
                             "p_value", "fdr") if c in ora_res.columns]
        print(f"  {len(ora_res):,} sets tested; top 10 by FDR:")
        print(ora_res.head(10)[cols].round(4).to_string(index=False))

# 4. Preranked GSEA on the moderated-t statistic (signed).
if libraries is not None:
    stat = "stat" if "stat" in results_canon.columns else "log2fc"
    ranked = (results_canon.set_index("site_id")[stat]
                            .groupby(level=0).max().dropna())
    print(f"\n[GSEA]  ranking on {stat!r}  ({len(ranked):,} unique sites, "
          f"1000 permutations)")
    gsea_res = gsea(ranked, libraries=libraries, n_permutations=1000, seed=42)
    if len(gsea_res):
        cols = [c for c in ("library", "set_name", "NES", "direction",
                             "p_value", "fdr") if c in gsea_res.columns]
        print("  Top 10 by FDR:")
        print(gsea_res.head(10)[cols].round(4).to_string(index=False))

# 5. Cross-connection to §7b: GSEA on PC1 loadings (signed).
if libraries is not None:
    pc_loadings = ap.dimred.loadings_for_enrichment(adata_pca)
    print(f"\n[GSEA on PC1 loadings]  {len(pc_loadings):,} canonical sites")
    gsea_pc1 = gsea(pc_loadings["PC1"], libraries=libraries,
                    n_permutations=1000, seed=42)
    if len(gsea_pc1):
        cols = [c for c in ("library", "set_name", "NES", "direction",
                             "p_value", "fdr") if c in gsea_pc1.columns]
        print("  Top 10 sets driving PC1:")
        print(gsea_pc1.head(10)[cols].round(4).to_string(index=False))


## §12c. Gene-level pathway enrichment (Enrichr / gseapy)

`ap.enrichment.pathway_enrichment` and `ap.enrichment.pathway_gsea` collapse
sites to genes and run **gene-level** enrichment against the standard
libraries (GO / KEGG / Reactome / Hallmark) via [gseapy](https://github.com/zqfang/GSEApy).
Complementary to §12b: **site-level PTM biology** (activating / inhibitory
sites, PPI regulation) vs **gene-level pathway biology** (which pathways
are perturbed).

- **`pathway_enrichment`** — ORA-style, Enrichr backend (network call).
  Fast, categorical, uses the community-curated Enrichr libraries.
- **`pathway_gsea`** — Preranked GSEA on the per-gene rank
  (sites → genes via `_collapse_max_abs` or `_collapse_top_significant`).
  Slower (permutation) but uses every gene's rank.

Both require `pip install alphaPhos[enrichment]` (pulls in gseapy).
`pathway_enrichment` also needs network access (Enrichr API).


In [ ]:
from alphaphos.enrichment import (
    pathway_enrichment, pathway_gsea, DEFAULT_LIBRARIES_HUMAN,
)

# In this notebook `results` has a `protein` column (not indexed by key).
# Both pathway functions accept key_column= to pick that up.
try:
    print(f"[pathway_enrichment]  libraries: {DEFAULT_LIBRARIES_HUMAN}")
    path_ora = pathway_enrichment(
        results,
        fdr_threshold=0.05,
        log2fc_threshold=0.585,
        libraries=DEFAULT_LIBRARIES_HUMAN,
        organism="human",
        key_column="protein",
        stat_col="log2fc",
        fdr_col="fdr",
    )
    if len(path_ora):
        cols = [c for c in ("library", "term", "n_overlap", "odds_ratio",
                             "adj_p", "genes") if c in path_ora.columns]
        print(f"  {len(path_ora):,} terms; top 10 by adj_p:")
        print(path_ora.head(10)[cols].round(4).to_string(index=False))
except Exception as e:
    print(f"Skipping pathway_enrichment: {type(e).__name__}: {e}")
    print("  (needs `pip install alphaPhos[enrichment]` and network access to Enrichr)")

try:
    stat = "stat" if "stat" in results.columns else "log2fc"
    print(f"\n[pathway_gsea]  preranked GSEA on {stat!r}")
    path_gsea = pathway_gsea(
        results,
        stat_col=stat,
        libraries=DEFAULT_LIBRARIES_HUMAN,
        site_to_gene_agg="max_abs",
        organism="human",
        key_column="protein",
        n_permutations=1000,
    )
    if len(path_gsea):
        cols = [c for c in ("library", "term", "NES", "p_value", "fdr",
                             "leading_edge") if c in path_gsea.columns]
        print(path_gsea.head(10)[cols].round(4).to_string(index=False))
except Exception as e:
    print(f"Skipping pathway_gsea: {type(e).__name__}: {e}")
    print("  (needs `pip install alphaPhos[enrichment]`)")


## §13. Dose-response analysis (CurveCurator)

**Use this section only if your experiment is a dose-response study**
(varying concentrations of a drug/ligand at one or more timepoints).
Skip if your study is a binary contrast (e.g. +EGF vs -EGF — already
analysed in §8).

`alphaphos.dose_response.fit_dose_response` wraps the
[CurveCurator](https://github.com/kusterlab/curve_curator) CLI (Kuster
lab, TUM) which fits **4-parameter log-logistic** dose-response curves
to MS data and reports per-curve quality (pEC50, fold change, R²,
target-decoy FDR q-value).

Inputs needed from `adata.obs`:
- **`dose_col`** (default `"dose"`) — numeric, in nM. **DMSO/control
  samples must have dose == 0.**
- **`timepoint_col`** (default `"timepoint"`) — pass `None` for
  single-timepoint experiments. For multi-timepoint, CurveCurator is
  run **independently per timepoint** (CurveCurator doesn't fit
  dose×time jointly) and the curves are concatenated.

The orchestrator:
1. Builds CurveCurator's TSV input (sites as rows, `Raw <sample>`
   columns with linear intensities — converted from log2 internally).
2. Generates the TOML config (experimental design, fit parameters).
3. Runs `python -m curve_curator --fdr config.toml` as a subprocess.
4. Parses `curves.txt` back into a tidy pandas DataFrame.

For each (site, timepoint) you get: `pEC50`, `EC50_nM`,
`Curve Fold Change`, `Curve R2`, `Curve F_Value`, `Curve P_Value`, and
(when `fdr=True`) `Curve q_Value` + `Curve Regulation` (the FDR-call).

The interactive dashboard HTML is written to each per-timepoint
output dir (open in browser to interactively browse curves).


In [ ]:
from pathlib import Path
from alphaphos.dose_response import fit_dose_response

# Pre-flight: only run this section if obs has a 'dose' column with a 0-dose group
if "dose" not in adata.obs.columns or (adata.obs["dose"] == 0).sum() == 0:
    print("Skipping §13 — adata.obs has no 'dose' column with DMSO (dose==0) samples.")
    print("This pipeline section is for dose-response experiments only.")
else:
    out_root = Path("D:/Projects/alphaPhos/test_data/cc_runs")
    curves = fit_dose_response(
        adata,
        output_root=out_root,
        dose_col="dose",
        timepoint_col="timepoint" if "timepoint" in adata.obs.columns else None,
        condition="drug",
        description="alphaPhos pipeline walkthrough",
        max_missing=12,       # tweak based on n_wells
        available_cores=4,
        fdr=True,
        verbose=True,
    )

    print(f"\nFit {len(curves):,} curves across {curves.get('timepoint', pd.Series([0])).nunique()} timepoint(s).")
    print("\nTop 15 regulated sites (smallest q-value):")
    cols = [c for c in ("site_key","timepoint","pEC50","EC50_nM",
                         "Curve Fold Change","Curve R2","Curve q_Value")
            if c in curves.columns]
    print(curves.sort_values("Curve q_Value").head(15)[cols].round(3))

    # Stash for downstream / sharing
    adata.uns["alphaphos_dose_response"] = curves.copy()


## §14. QC dashboard (one-call HTML)

`alphaphos.qc.generate_dashboard(adata, output)` writes a single,
self-contained, interactive HTML file with all phospho-aware QC panels:

- **§1 Pipeline waterfall** — row counts at each filter step
- **§2 Sample QC grid** — S/T/Y composition, localization distribution,
  multiplicity, missingness, n_classI, per-condition CV by AA
- **§3 Reproducibility** — replicate correlation matrix (Pearson r),
  hierarchically clustered for sample order
- **§4 Class I + contaminants** — per-cell binary vs condition-aware
  policy comparison, optional contaminant breakdown
- **§5 Imputation** — MAR vs MNAR cells per sample (when hybrid imputer
  audit is provided)
- **§6 Provenance** — pipeline parameters from `adata.uns['alphaphos']`

Built on bokeh — every plot is hover/zoom/pan interactive. Open the
output HTML in any browser, share, or archive.


In [19]:
from pathlib import Path
import importlib
import alphaphos.qc, alphaphos.qc.plots, alphaphos.qc.metrics, alphaphos.qc.dashboard
importlib.reload(alphaphos.qc.plots)
importlib.reload(alphaphos.qc.metrics)
importlib.reload(alphaphos.qc.dashboard)
importlib.reload(alphaphos.qc)
from alphaphos.qc import generate_dashboard


out_path = Path("D:/Projects/alphaPhos/test_data/qc_dashboard.html")

# Optional: pass the raw PSM df to enable the contaminant panel.
# (we already used df_raw with drop_contaminants=False if you have it;
# otherwise leave psm_df=None)
psm_df_optional = None  # or: read_psm(PSM_TSV, drop_contaminants=False, ...)

# Optional: pass impute_hybrid's audit DataFrame to enable the MAR/MNAR panel.
# Re-run impute with return_audit=True if you want this.
impute_audit_optional = None

dashboard_path = generate_dashboard(
    adata,
    output_path=out_path,
    psm_df=psm_df_optional,
    impute_audit=impute_audit_optional,
    title="alphaPhos QC | EGF nanoPhos benchmark",
)
print(f"Wrote dashboard: {dashboard_path}  ({dashboard_path.stat().st_size:,} bytes)")
print("Open it in your browser to inspect interactively.")


D:\Projects\alphaPhos\src\alphaphos\qc\plots.py:336: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_long = corr_df.stack(dropna=False).reset_index()



Wrote dashboard: D:\Projects\alphaPhos\test_data\qc_dashboard.html  (1,553,436 bytes)
Open it in your browser to inspect interactively.


## §14b. Cross-species orthology mapping (`alphaphos.orthology`)

`ap.orthology.map_to_human` maps each site in `adata.var` to its best-match
human ortholog site by **flanking-sequence window matching**. Works
against any species FASTA — the module has been validated on the mouse
SwissProt (60.2% mapping rate; per-species-entrapment-FDR ≤ 5% at
`max_mismatches=2`).

Method:
1. Extract `±window_size` flanking windows around every human S/T/Y from the
   human FASTA and build a segment-based lookup index (pigeonhole principle
   → tolerates up to `max_mismatches` substitutions).
2. For each source site, look up the flanking window against the index.
3. Score candidates with **target-decoy FDR** (Elias-Gygi 2007) using
   reversed-window decoys.
4. Tiebreak by (a) fewer mismatches (b) matching gene name (c) matching
   residue class (S/T vs Y).
5. Optional broader-window verification pass (`verify_window_size=30`) that
   re-scores the top candidates for paralog disambiguation.

**Note:** the EGF dataset is human, so this is a **no-op demo** — sites
map to themselves. Swap in a mouse or rat AnnData and you get a
production-quality cross-species phospho-site translation.


In [ ]:
from pathlib import Path
import alphaphos as ap

HUMAN_FASTA = ROOT / "resources" / "fastas" / "human.fasta"
MOUSE_FASTA = ROOT / "resources" / "fastas" / "mouse.fasta"

# The kinase_sequence column is required.  If §3 ran without fasta_path, add
# it now via ap.add_kinase_windows on the human FASTA.
if "kinase_sequence" not in adata.var.columns:
    try:
        adata = ap.add_kinase_windows(adata, fasta_path=HUMAN_FASTA)
        print(f"Added kinase_sequence for {adata.var['kinase_sequence'].notna().sum():,} sites.")
    except Exception as e:
        print(f"Cannot demo orthology: {type(e).__name__}: {e}")

if "kinase_sequence" in adata.var.columns:
    slice_adata = adata[:, :200].copy()
    try:
        mapped = ap.orthology.map_to_human(
            slice_adata,
            human_fasta=HUMAN_FASTA,
            # source_fasta=HUMAN_FASTA,  # only for verify_window_size pass
            copy=True,
        )
        orth_cols = [
            c for c in ("human_gene", "human_uniprot", "human_site", "human_site_key",
                        "mismatches", "mapping_qvalue", "ortholog_ambiguous")
            if c in mapped.var.columns
        ]
        n_mapped = int(mapped.var["human_site_key"].notna().sum())
        print(f"[orthology.map_to_human] mapped {n_mapped}/{slice_adata.n_vars} sites")
        print(f"  new .var columns: {orth_cols}")

        if n_mapped:
            show_cols = [c for c in ("gene_first", "site_aa", "site_position",
                                      "human_site_key", "human_gene", "mismatches",
                                      "mapping_qvalue") if c in mapped.var.columns]
            subset = mapped.var.loc[mapped.var["human_site_key"].notna(), show_cols].head(10)
            print("\nFirst 10 mapped sites (human -> human, identity demo):")
            print(subset.to_string())
    except Exception as e:
        print(f"orthology.map_to_human hit an error: {type(e).__name__}: {e}")


## §15. Save the analysed AnnData

`adata.write_h5ad` persists everything in one HDF5 file:
- `.X` (imputed log2 quants)
- `.obs` (sample metadata)
- `.var` (per-site metadata: structural + motif flags + site QC)
- `.layers["intensity_log2"]` (the imputed quants)
- `.layers["localization"]` (per-site, per-sample loc probs)
- `.uns["alphaphos"]` (pipeline provenance)
- `.uns["pca"]` (PCA result)
- `.obsm["X_pca"]` (PCA embedding)
- `.uns["classI_decision_table"]` (the per-condition Class-I decision table)

Differential results live in their own DataFrame — save alongside.


In [ ]:
adata_path = OUT_DIR / "pipeline_walkthrough_adata.h5ad"
adata.write_h5ad(adata_path)
print(f"AnnData saved to: {adata_path}")
print(f"  size: {adata_path.stat().st_size / 1e6:.1f} MB")

results_path = OUT_DIR / "pipeline_walkthrough_results.parquet"
results.to_parquet(results_path)
print(f"Differential results saved to: {results_path}")
print(f"  size: {results_path.stat().st_size / 1e6:.1f} MB")


## What you have at this point

- `adata` — full quantitative matrix with phospho-specific metadata, ready
  for any scverse-compatible downstream tool
- `results` — differential analysis table (1 row per site)
- `adata_pca` — sample-level PCA results with loadings + variance ratios
  attached (from §7b `ap.dimred.pca`)
- Two persisted files: `pipeline_walkthrough_adata.h5ad` +
  `pipeline_walkthrough_results.parquet`
- Enrichment DataFrames from §11, §12, §12b, §12c: KSEA (Fisher +
  MEA + network ULM), PTM-DB ORA/GSEA, gene-level pathway enrichment.
- Ortholog-mapped `.var` columns from §14b (identity mapping on human
  data; production feature is human ↔ mouse/rat/etc.).

## Modules exercised

| Module | Where |
|---|---|
| `alphaphos.io.read_spectronaut` | §1 |
| `alphaphos.preprocess.collapse_sites / to_anndata / impute_hybrid` | §3, §4, §6 |
| `alphaphos.kinase` (Yaffe PWM predict + Fisher/MEA KSEA) | §10, §11 |
| `alphaphos.ksea` (network-based KSEA, decoupler) | §12 |
| **`alphaphos.dimred`** (PCA + loadings + imputation-impact) | **§7b** |
| **`alphaphos.enrichment`** (PTM-DB ORA/GSEA, pathway) | **§12b, §12c** |
| **`alphaphos.orthology`** (cross-species site mapping) | **§14b** |
| `alphaphos.dose_response` | §13 |
| `alphaphos.qc.generate_dashboard` | §14 |
| `apt.pp.filter_data_completeness` | §5 |
| `apt.tl.pca` (legacy — kept alongside `ap.dimred` in §7b) | §7 |
| `apt.tl.diff_exp_ebayes` + `apt.pl.volcano` | §8, §9 |

See **`docs/benchmark/spectronaut_native_benchmark.md`** for the full validation
of every default in this pipeline against Spectronaut's native classI PTM site
report.
